# LeetCode 2133 — Check if Every Row and Column Contains All Numbers bằng Python

**Link đề:** https://leetcode.com/problems/check-if-every-row-and-column-contains-all-numbers/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Matrix

```python
class Solution:
    def checkValid(self, matrix: list[list[int]]) -> bool:
        ...
```

## Mục tiêu học
1. Đọc và duyệt **ma trận** (list lồng list): hàng `matrix[i]`, cột `matrix[r][j]`, kích thước `n`.
2. Đi từ cách "hỏi từng số 1..n có trong hàng không" `O(n³)` → dùng **`set`** để kiểm tra trong `O(n²)`.
3. Hiểu `set`, `len(set(...))`, so sánh hai set, `zip(*matrix)` để lấy cột, `all(...)`.
4. Trace từng hàng/cột: in set đang xây và quyết định dừng sớm.
5. Biết vì sao `len(set(row)) == n` **đủ** nhờ constraint `1 <= matrix[i][j] <= n`, và khi nào nó **không đủ**.

## 1. Đề bài

Ma trận `n x n` được gọi là **hợp lệ (valid)** nếu **mỗi hàng** và **mỗi cột** đều chứa **đủ tất cả** các số nguyên từ `1` đến `n`.

Cho ma trận `matrix` kích thước `n x n`, trả `True` nếu hợp lệ, ngược lại trả `False`.

| `matrix` | Kết quả | Vì sao |
|---|:---:|---|
| `[[1,2,3],[3,1,2],[2,3,1]]` | `True` | mọi hàng, mọi cột đều là một hoán vị của `{1,2,3}` |
| `[[1,1,1],[1,2,3],[1,2,3]]` | `False` | hàng 0 chỉ có `{1}`, cột 0 cũng chỉ có `{1}` — thiếu 2 và 3 |

**Constraints:**
- `n == matrix.length == matrix[i].length`
- `1 <= n <= 100`
- `1 <= matrix[i][j] <= n`

Đây chính là điều kiện của một **hình vuông Latin (Latin square)** — giống luật Sudoku nhưng không có ô 3x3.

Câu hỏi then chốt: *"Làm sao biết một dãy `n` số có chứa đủ `1..n` hay không, một cách nhanh và chắc chắn?"*

In [1]:
matrix = [[1, 2, 3],
          [3, 1, 2],
          [2, 3, 1]]
n = len(matrix)
print("n =", n)
for row in matrix:
    print(row)

n = 3
[1, 2, 3]
[3, 1, 2]
[2, 3, 1]


## 2. Nền tảng Python cần biết

### 2.1. Ma trận = list chứa các list

- `matrix[i]` là **hàng** thứ `i` (một list).
- `matrix[i][j]` là ô ở hàng `i`, cột `j`.
- `len(matrix)` là số hàng; vì ma trận vuông nên cũng là số cột `n`.

Python **không** có sẵn "cột" như một list — muốn lấy cột `j` ta phải đi qua mọi hàng và lấy phần tử thứ `j`.

In [2]:
matrix = [[1, 2, 3],
          [3, 1, 2],
          [2, 3, 1]]
print("Hàng 1:", matrix[1])
print("Ô (1, 2):", matrix[1][2])
col_0 = [matrix[r][0] for r in range(len(matrix))]
print("Cột 0:", col_0)

Hàng 1: [3, 1, 2]
Ô (1, 2): 2
Cột 0: [1, 3, 2]


### 2.2. List comprehension `[biểu_thức for x in dãy]`

`[matrix[r][0] for r in range(n)]` đọc là: *"với mỗi `r` từ 0 đến n-1, lấy `matrix[r][0]` rồi gom vào list mới"*. Nó tương đương vòng `for` + `append`, chỉ ngắn hơn.

In [3]:
n = 3
squares_loop = []
for x in range(n):
    squares_loop.append(x * x)
squares_comp = [x * x for x in range(n)]
print(squares_loop, squares_comp, squares_loop == squares_comp)

[0, 1, 4] [0, 1, 4] True


### 2.3. `set` — tập hợp không trùng lặp

- `set(dãy)` tạo tập hợp: **bỏ trùng**, không quan tâm thứ tự.
- `x in s` kiểm tra tồn tại, trung bình `O(1)` (nhờ băm — hashing).
- `len(s)` = số giá trị **khác nhau**.
- `s.add(x)` thêm một phần tử.

In [4]:
row = [1, 1, 1]
s = set(row)
print("set([1,1,1]) =", s, "| len =", len(s))
s2 = set([3, 1, 2])
print("set([3,1,2]) =", s2, "| len =", len(s2))
print("2 in s2:", 2 in s2, "| 5 in s2:", 5 in s2)
s.add(7)
print("sau khi add 7:", s)

set([1,1,1]) = {1} | len = 1
set([3,1,2]) = {1, 2, 3} | len = 3
2 in s2: True | 5 in s2: False
sau khi add 7: {1, 7}


### 2.4. So sánh hai set bằng `==`

Hai set bằng nhau khi chứa **đúng cùng các phần tử**, bất kể thứ tự. `set(range(1, n + 1))` chính là tập "mục tiêu" `{1, 2, ..., n}`.

In [5]:
n = 3
target = set(range(1, n + 1))
print("target =", target)
print(set([2, 3, 1]) == target)   # True
print(set([1, 1, 3]) == target)   # False: thiếu 2

target = {1, 2, 3}
True
False


### 2.5. `zip(*matrix)` — lật hàng thành cột

- `*matrix` "rải" các hàng thành các đối số riêng: `zip([1,2,3], [3,1,2], [2,3,1])`.
- `zip` ghép phần tử thứ 0 của mọi hàng thành một tuple, rồi phần tử thứ 1, ... → mỗi tuple là **một cột**.

Đây là cách viết chuyển vị (transpose) ma trận rất hay gặp trong Python.

In [6]:
matrix = [[1, 2, 3],
          [3, 1, 2],
          [2, 3, 1]]
for j, col in enumerate(zip(*matrix)):
    print("cột", j, "=", col)

cột 0 = (1, 3, 2)
cột 1 = (2, 1, 3)
cột 2 = (3, 2, 1)


### 2.6. `all(...)` và `enumerate`

- `all(iterable)` trả `True` nếu **mọi** phần tử đều truthy; dừng ngay khi gặp `False` đầu tiên (short-circuit).
- `enumerate(dãy)` cho cặp `(chỉ_số, giá_trị)` — tiện để in "hàng thứ mấy".

In [7]:
print(all([True, True, True]))
print(all([True, False, True]))
print(all(x > 0 for x in [3, 1, 2]))
for i, v in enumerate(["a", "b"]):
    print(i, v)

True
False
True
0 a
1 b


### 2.7. Hiểu bài bằng tay

Với `n = 3`, mỗi hàng phải là một **hoán vị** của `{1,2,3}`, ví dụ `[3,1,2]`. Hàng `[1,1,3]` hỏng vì lặp `1` → chắc chắn thiếu một số khác.

Quan sát quan trọng: mỗi hàng có **đúng n ô**, mỗi ô là một số trong `1..n` (constraint). Vậy:

> Hàng chứa đủ `1..n` ⟺ `n` ô đôi một **khác nhau** ⟺ `len(set(row)) == n`.

Lý do: có `n` "chuồng" (giá trị 1..n) và `n` "chim" (ô). Nếu không trùng thì mỗi chuồng có đúng một chim → đủ hết. Nếu trùng thì theo **nguyên lý Dirichlet (chuồng bồ câu)** có chuồng trống → thiếu số.

In [8]:
n = 3
for row in ([3, 1, 2], [1, 1, 3], [2, 2, 2]):
    print(row, "-> số khác nhau:", len(set(row)), "| đủ 1..n?", set(row) == set(range(1, n + 1)))

[3, 1, 2] -> số khác nhau: 3 | đủ 1..n? True
[1, 1, 3] -> số khác nhau: 2 | đủ 1..n? False
[2, 2, 2] -> số khác nhau: 1 | đủ 1..n? False


## 3. Cách 1 — Trực giác: hỏi từng số `1..n` có mặt không

Người mới thường làm đúng theo lời đề: với mỗi hàng, với mỗi số `v` từ `1` đến `n`, kiểm tra `v in row`. Làm tương tự cho mỗi cột.

`v in row` trên **list** phải dò từng phần tử → tốn `O(n)`.

In [9]:
def check_valid_naive(matrix: list[list[int]]) -> bool:
    n = len(matrix)
    for i in range(n):
        row = matrix[i]
        col = [matrix[r][i] for r in range(n)]
        for v in range(1, n + 1):
            if v not in row or v not in col:
                return False
    return True

print(check_valid_naive([[1, 2, 3], [3, 1, 2], [2, 3, 1]]))  # True
print(check_valid_naive([[1, 1, 1], [1, 2, 3], [1, 2, 3]]))  # False

True
False


### Chi phí của cách trực giác

- Có `n` hàng (+ `n` cột), mỗi cái hỏi `n` số, mỗi lần hỏi `v in list` tốn tới `n` bước → **`O(n³)`**.
- Với `n = 100`: khoảng `2 · 100³ = 2.000.000` phép so sánh — vẫn chạy được, nhưng ta làm tốt hơn được.

Đếm thử số phép so sánh ở trường hợp xấu (ma trận hợp lệ, phải xét hết):

In [10]:
def count_naive_ops(matrix):
    n = len(matrix)
    ops = 0
    for i in range(n):
        row = matrix[i]
        col = [matrix[r][i] for r in range(n)]
        for v in range(1, n + 1):
            ops += row.index(v) + 1      # số bước dò trong hàng
            ops += col.index(v) + 1      # số bước dò trong cột
    return ops

for n in (3, 10, 50):
    m = [[(r + c) % n + 1 for c in range(n)] for r in range(n)]   # ma trận hợp lệ dạng dịch vòng
    print(f"n={n:>3}: ~{count_naive_ops(m)} phép so sánh, n^3 = {n**3}")

n=  3: ~36 phép so sánh, n^3 = 27
n= 10: ~1100 phép so sánh, n^3 = 1000
n= 50: ~127500 phép so sánh, n^3 = 125000


## 4. Nút thắt và insight

**Nút thắt:** câu hỏi "`v` có trong list không?" bị lặp lại `n` lần trên cùng một hàng, mỗi lần dò lại từ đầu.

**Insight 1:** đổ hàng vào `set` **một lần** (`O(n)`), sau đó mọi câu hỏi tồn tại đều `O(1)` trung bình.

**Insight 2 (nhờ constraint):** thậm chí không cần hỏi từng `v` — chỉ cần `len(set(row)) == n`. Vì mọi giá trị đều nằm trong `1..n`, "không trùng" đồng nghĩa "đủ".

Mỗi hàng/cột tốn `O(n)` → tổng `2n` dãy × `O(n)` = **`O(n²)`**, tức là tỷ lệ với số ô — không thể nhanh hơn vì phải đọc mọi ô ít nhất một lần.

In [11]:
row = [3, 1, 2]
s = set(row)          # xây 1 lần
print([v in s for v in range(1, 4)])   # mỗi câu hỏi O(1)
print("len(set(row)) == 3 ?", len(s) == 3)

[True, True, True]
len(set(row)) == 3 ? True


## 5. Cách 2 — Lời giải chính: set cho từng hàng và từng cột

Ta giảng từng đoạn trước khi ghép.

**Đoạn 1 — kiểm tra một dãy:**
```python
def is_full(seq, n):
    return len(set(seq)) == n
```

**Đoạn 2 — duyệt hàng:** `for row in matrix: if not is_full(row, n): return False`

**Đoạn 3 — duyệt cột:** cột `j` là `[matrix[r][j] for r in range(n)]`.

**Đoạn 4 — vượt qua hết thì `return True`.** Dừng sớm ngay khi gặp hàng/cột hỏng.

In [12]:
def is_full(seq, n):
    return len(set(seq)) == n

print(is_full([3, 1, 2], 3), is_full([1, 1, 3], 3))

True False


In [13]:
def check_valid_set(matrix: list[list[int]]) -> bool:
    n = len(matrix)
    # Đoạn 2: mỗi hàng
    for row in matrix:
        if len(set(row)) != n:
            return False
    # Đoạn 3: mỗi cột
    for j in range(n):
        col = [matrix[r][j] for r in range(n)]
        if len(set(col)) != n:
            return False
    return True

print(check_valid_set([[1, 2, 3], [3, 1, 2], [2, 3, 1]]))  # True
print(check_valid_set([[1, 1, 1], [1, 2, 3], [1, 2, 3]]))  # False
print(check_valid_set([[1]]))                              # True

True
False
True


## 6. TRACE từng bước

Ví dụ khó hơn một chút: mọi **hàng** đều hợp lệ nhưng có **cột** hỏng — để thấy vì sao phải kiểm cả cột.

`matrix = [[1,2,3],[1,2,3],[3,1,2]]`

| Bước | Loại | Dãy | `set` | `len` | Quyết định |
|---|---|---|---|:-:|---|
| 1 | hàng 0 | `[1,2,3]` | `{1,2,3}` | 3 | ok, đi tiếp |
| 2 | hàng 1 | `[1,2,3]` | `{1,2,3}` | 3 | ok |
| 3 | hàng 2 | `[3,1,2]` | `{1,2,3}` | 3 | ok |
| 4 | cột 0 | `[1,1,3]` | `{1,3}` | 2 | **≠ 3 → return False** |

Cột: hàng nào cũng đúng, nhưng hai hàng đầu giống hệt nhau nên cột 0 lặp số `1`.

In [14]:
def trace_check_valid(matrix):
    n = len(matrix)
    step = 1
    for i, row in enumerate(matrix):
        s = set(row)
        print(f"Bước {step}: hàng {i} = {row} -> set = {sorted(s)}, len = {len(s)}")
        step += 1
        if len(s) != n:
            print("  -> thiếu số, return False")
            return False
    for j in range(n):
        col = [matrix[r][j] for r in range(n)]
        s = set(col)
        print(f"Bước {step}: cột {j} = {col} -> set = {sorted(s)}, len = {len(s)}")
        step += 1
        if len(s) != n:
            print("  -> thiếu số, return False")
            return False
    print("Mọi hàng và cột đều đủ 1..n -> return True")
    return True

trace_check_valid([[1, 2, 3], [1, 2, 3], [3, 1, 2]])

Bước 1: hàng 0 = [1, 2, 3] -> set = [1, 2, 3], len = 3
Bước 2: hàng 1 = [1, 2, 3] -> set = [1, 2, 3], len = 3
Bước 3: hàng 2 = [3, 1, 2] -> set = [1, 2, 3], len = 3
Bước 4: cột 0 = [1, 1, 3] -> set = [1, 3], len = 2
  -> thiếu số, return False


False

In [15]:
trace_check_valid([[1, 2, 3], [3, 1, 2], [2, 3, 1]])

Bước 1: hàng 0 = [1, 2, 3] -> set = [1, 2, 3], len = 3
Bước 2: hàng 1 = [3, 1, 2] -> set = [1, 2, 3], len = 3
Bước 3: hàng 2 = [2, 3, 1] -> set = [1, 2, 3], len = 3
Bước 4: cột 0 = [1, 3, 2] -> set = [1, 2, 3], len = 3
Bước 5: cột 1 = [2, 1, 3] -> set = [1, 2, 3], len = 3
Bước 6: cột 2 = [3, 2, 1] -> set = [1, 2, 3], len = 3
Mọi hàng và cột đều đủ 1..n -> return True


True

## 7. Code nộp LeetCode — Cách 2

Gọn, đúng signature `checkValid(self, matrix)`.

In [16]:
class Solution:
    def checkValid(self, matrix: list[list[int]]) -> bool:
        n = len(matrix)
        for row in matrix:
            if len(set(row)) != n:
                return False
        for j in range(n):
            if len({matrix[r][j] for r in range(n)}) != n:
                return False
        return True

sol = Solution()
print(sol.checkValid([[1, 2, 3], [3, 1, 2], [2, 3, 1]]))
print(sol.checkValid([[1, 1, 1], [1, 2, 3], [1, 2, 3]]))

True
False


Chú ý `{matrix[r][j] for r in range(n)}` là **set comprehension** — giống list comprehension nhưng dùng ngoặc nhọn, tạo thẳng set.

## 8. Cách 3 — Pythonic: `zip(*matrix)` + so sánh với tập mục tiêu

Ý tưởng giống Cách 2, nhưng:
- Lấy cột bằng `zip(*matrix)` thay vì tự dựng.
- So sánh `set(line) == target` — **không phụ thuộc** vào constraint "giá trị nằm trong 1..n", nên an toàn hơn nếu đề đổi.
- Gộp hàng và cột: `itertools.chain(matrix, zip(*matrix))` hoặc đơn giản `matrix + list(zip(*matrix))`.

In [17]:
def check_valid_zip(matrix: list[list[int]]) -> bool:
    n = len(matrix)
    target = set(range(1, n + 1))
    lines = list(matrix) + list(zip(*matrix))   # n hàng + n cột
    return all(set(line) == target for line in lines)

print(check_valid_zip([[1, 2, 3], [3, 1, 2], [2, 3, 1]]))
print(check_valid_zip([[1, 2, 3], [1, 2, 3], [3, 1, 2]]))

True
False


In [18]:
class Solution:
    def checkValid(self, matrix: list[list[int]]) -> bool:
        target = set(range(1, len(matrix) + 1))
        return all(set(row) == target for row in matrix) and \
               all(set(col) == target for col in zip(*matrix))

print(Solution().checkValid([[1, 2, 3], [3, 1, 2], [2, 3, 1]]))

True


## 9. Cách 4 — Mảng đánh dấu (boolean array / bitmask), một lần quét

Thay vì set, dùng mảng `seen` kích thước `n + 1` (chỉ số = giá trị). Vì giá trị ∈ `1..n`, ta đánh dấu `seen[v] = True`; nếu gặp lại ô đã đánh dấu → trùng → `False`.

Hay hơn nữa: duyệt **một lần** qua cả ma trận, mỗi hàng `i` và mỗi cột `j` có một mảng đánh dấu riêng. Ta có thể dùng **bitmask** (một số nguyên, bit thứ `v` = đã thấy `v`):
- `mask >> v & 1` đọc bit `v`.
- `mask |= 1 << v` bật bit `v`.

In [19]:
mask = 0
for v in [3, 1]:
    mask |= 1 << v
    print(f"bật bit {v}: mask = {mask} = {bin(mask)}")
print("đã thấy 3?", mask >> 3 & 1, "| đã thấy 2?", mask >> 2 & 1)

bật bit 3: mask = 8 = 0b1000
bật bit 1: mask = 10 = 0b1010
đã thấy 3? 1 | đã thấy 2? 0


In [20]:
def check_valid_bitmask(matrix: list[list[int]]) -> bool:
    n = len(matrix)
    row_mask = [0] * n
    col_mask = [0] * n
    for i in range(n):
        for j in range(n):
            bit = 1 << matrix[i][j]
            if row_mask[i] & bit or col_mask[j] & bit:
                return False          # giá trị lặp trong hàng i hoặc cột j
            row_mask[i] |= bit
            col_mask[j] |= bit
    return True

print(check_valid_bitmask([[1, 2, 3], [3, 1, 2], [2, 3, 1]]))
print(check_valid_bitmask([[1, 2, 3], [1, 2, 3], [3, 1, 2]]))

True
False


Cách 4 cũng dựa vào lập luận chuồng bồ câu: không lặp trong `n` ô với giá trị `1..n` ⇒ đủ. Code nộp:

In [21]:
class Solution:
    def checkValid(self, matrix: list[list[int]]) -> bool:
        n = len(matrix)
        row_mask = [0] * n
        col_mask = [0] * n
        for i in range(n):
            for j in range(n):
                bit = 1 << matrix[i][j]
                if row_mask[i] & bit or col_mask[j] & bit:
                    return False
                row_mask[i] |= bit
                col_mask[j] |= bit
        return True

print(Solution().checkValid([[1]]), Solution().checkValid([[1, 2], [2, 1]]), Solution().checkValid([[1, 2], [1, 2]]))

True True False


## 10. Vì sao đúng? (invariant)

**Bổ đề:** một dãy `L` gồm đúng `n` số, mỗi số thuộc `{1..n}`. Khi đó `L` chứa đủ `1..n` ⟺ các phần tử của `L` đôi một khác nhau ⟺ `len(set(L)) == n`.
- (⇒) Đủ `n` giá trị khác nhau trong `n` ô → mỗi giá trị đúng một lần.
- (⇐) `n` giá trị khác nhau, đều thuộc tập có `n` phần tử → phủ kín tập đó.

**Invariant Cách 2:** trước khi xét hàng/cột thứ `t`, mọi dãy đã xét đều hợp lệ. Nếu gặp dãy hỏng, đề bài đã sai → trả `False` là đúng. Nếu xét hết `2n` dãy mà không hỏng → đúng định nghĩa "mọi hàng và mọi cột" → `True`.

**Invariant Cách 4:** sau khi xử lý ô `(i, j)`, `row_mask[i]` chứa đúng tập giá trị của các ô đã đọc trong hàng `i` (tương tự cột `j`). Gặp bit đã bật ⇔ hàng/cột có giá trị lặp.

**Cách 3** không cần bổ đề vì so sánh trực tiếp với `{1..n}`.

## 11. Độ phức tạp

Gọi `n` là cạnh ma trận (có `n²` ô).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Hỏi từng `v in list` | `O(n³)` | `O(n)` (list cột) | dò list mỗi lần |
| 2. `len(set)` theo hàng/cột | `O(n²)` trung bình | `O(n)` | set của một dãy |
| 3. `zip(*matrix)` + target | `O(n²)` trung bình | `O(n²)` (`list(zip(*matrix))` chép cả ma trận) | gọn, không phụ thuộc constraint |
| 4. Bitmask một lần quét | `O(n²)` | `O(n)` mask (mỗi mask ~n bit) | không cần hash |

`O(n²)` là tối ưu vì phải đọc mỗi ô ít nhất một lần (một ô sai cũng làm thay đổi đáp án).

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

Các nhóm:
1. Hai ví dụ của đề.
2. `n = 1`: `[[1]]` luôn hợp lệ.
3. Hàng đúng hết nhưng cột sai (hai hàng giống nhau).
4. Cột đúng hết nhưng hàng sai (chuyển vị của ca 3).
5. Ô sai nằm ở **hàng/cột cuối** — để chắc vòng lặp đi đến cuối.

In [22]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<32} PASS ({actual})")

impls = {
    "naive": check_valid_naive,
    "set": check_valid_set,
    "zip": check_valid_zip,
    "bitmask": check_valid_bitmask,
    "Solution": Solution().checkValid,
}
cases = [
    ("đề ví dụ 1", [[1, 2, 3], [3, 1, 2], [2, 3, 1]], True),
    ("đề ví dụ 2", [[1, 1, 1], [1, 2, 3], [1, 2, 3]], False),
    ("n = 1", [[1]], True),
    ("hàng đúng, cột sai", [[1, 2, 3], [1, 2, 3], [3, 1, 2]], False),
    ("cột đúng, hàng sai", [[1, 1, 3], [2, 2, 1], [3, 3, 2]], False),
    ("n = 2 hợp lệ", [[2, 1], [1, 2]], True),
    ("sai ở ô cuối", [[1, 2, 3], [2, 3, 1], [3, 1, 1]], False),
]
for name, m, exp in cases:
    for impl_name, f in impls.items():
        assert f(m) == exp, (name, impl_name)
    check(name, impls["set"](m), exp)
print("Tất cả cách cho cùng kết quả!")

đề ví dụ 1                       PASS (True)
đề ví dụ 2                       PASS (False)
n = 1                            PASS (True)
hàng đúng, cột sai               PASS (False)
cột đúng, hàng sai               PASS (False)
n = 2 hợp lệ                     PASS (True)
sai ở ô cuối                     PASS (False)
Tất cả cách cho cùng kết quả!


### Kiểm thử ngẫu nhiên có seed

Sinh ma trận ngẫu nhiên nhỏ (rất hay ra `False`) và ma trận Latin hợp lệ (dịch vòng rồi xáo hàng/cột), đối chiếu mọi cách với cách trực giác (oracle).

In [23]:
import random
rng = random.Random(2133)

def random_latin(n):
    base = [[(r + c) % n + 1 for c in range(n)] for r in range(n)]
    rng.shuffle(base)                       # xáo hàng: vẫn hợp lệ
    perm = list(range(n)); rng.shuffle(perm)
    return [[row[p] for p in perm] for row in base]   # xáo cột: vẫn hợp lệ

checked = 0
for _ in range(300):
    n = rng.randint(1, 6)
    if rng.random() < 0.5:
        m = random_latin(n)
    else:
        m = [[rng.randint(1, n) for _ in range(n)] for _ in range(n)]
    expected = check_valid_naive(m)
    for f in impls.values():
        assert f(m) == expected
    checked += 1
print("Đã đối chiếu", checked, "ma trận ngẫu nhiên: PASS")

Đã đối chiếu 300 ma trận ngẫu nhiên: PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra hàng, quên cột

In [24]:
def wrong_rows_only(matrix):
    n = len(matrix)
    return all(len(set(row)) == n for row in matrix)

m = [[1, 2, 3], [1, 2, 3], [1, 2, 3]]
print("wrong_rows_only:", wrong_rows_only(m), "| đúng:", check_valid_set(m))

wrong_rows_only: True | đúng: False


**Vì sao sai:** đề yêu cầu *cả* hàng *và* cột. Ma trận ba hàng giống hệt nhau qua được kiểm tra hàng nhưng mỗi cột toàn một số. **Sửa:** thêm vòng kiểm tra cột.

### Lỗi 2: Kiểm tra `sum(row) == n*(n+1)//2` thay cho "đủ số"

In [25]:
def wrong_sum(matrix):
    n = len(matrix)
    need = n * (n + 1) // 2
    return all(sum(r) == need for r in matrix) and all(sum(c) == need for c in zip(*matrix))

m = [[2, 2, 2], [2, 2, 2], [2, 2, 2]]
print("tổng mỗi hàng =", sum(m[0]), "| wrong_sum:", wrong_sum(m), "| đúng:", check_valid_set(m))

tổng mỗi hàng = 6 | wrong_sum: True | đúng: False


**Vì sao sai:** tổng bằng nhau không suy ra tập bằng nhau: `2+2+2 = 1+2+3 = 6`. Đây là **phản ví dụ** kinh điển. **Sửa:** dùng set (hoặc đếm), không dùng tổng.

### Lỗi 3: Dùng `len(set(row)) == n` khi giá trị *không* bị giới hạn trong `1..n`

Nếu đề biến thể cho phép số bất kỳ, hàng `[1, 2, 5]` với `n = 3` có 3 giá trị khác nhau nhưng thiếu số `3`.

In [26]:
row = [1, 2, 5]
print("len(set) == 3:", len(set(row)) == 3, "| set == {1,2,3}:", set(row) == {1, 2, 3})

len(set) == 3: True | set == {1,2,3}: False


**Sửa:** khi không có constraint `1 <= x <= n`, so sánh `set(row) == set(range(1, n+1))` (Cách 3). Trong bài này constraint có, nên cả hai đều đúng — nhưng phải *biết vì sao*.

### Lỗi 4: Lấy cột sai chỉ số `matrix[j]` thay vì `matrix[r][j]`

In [27]:
matrix = [[1, 2, 3], [3, 1, 2], [2, 3, 1]]
j = 0
print("matrix[j]  (đây là HÀNG 0):", matrix[j])
print("cột 0 đúng:", [matrix[r][j] for r in range(len(matrix))])

matrix[j]  (đây là HÀNG 0): [1, 2, 3]
cột 0 đúng: [1, 3, 2]


### Lỗi 5: Quên rằng `zip(*matrix)` là iterator dùng một lần

`cols = zip(*matrix)` rồi duyệt hai lần → lần hai rỗng, `all([])` trả `True` một cách "ngây thơ". **Sửa:** `cols = list(zip(*matrix))` nếu cần dùng lại.

In [28]:
cols = zip(*[[1, 1], [1, 1]])
print("lần 1:", list(cols))
print("lần 2:", list(cols), "-> all([]) =", all([]))

lần 1: [(1, 1), (1, 1)]
lần 2: [] -> all([]) = True


## 14. Mẹo nhận dạng pattern

- "**Chứa đủ / mỗi số đúng một lần / không trùng**" → nghĩ ngay tới **set** hoặc **mảng đếm**.
- Giá trị bị chặn trong `1..n` và có đúng `n` phần tử → **nguyên lý chuồng bồ câu**: "không trùng ⟺ đủ".
- "Hàng **và** cột" → hai vòng, hoặc `zip(*matrix)`, hoặc mảng đánh dấu `row_seen[i]`, `col_seen[j]` một lần quét (giống **Valid Sudoku #36**).
- Đừng dùng **tổng** để kiểm tra tập hợp — luôn có phản ví dụ.

Bài liên quan: #36 Valid Sudoku, #217 Contains Duplicate, #268 Missing Number, #442/#448 (mảng giá trị 1..n).

## 15. Biến thể

### 15.1. Thêm cả hai đường chéo (Latin square "chéo")

In [29]:
def check_valid_with_diagonals(matrix):
    n = len(matrix)
    target = set(range(1, n + 1))
    main = {matrix[i][i] for i in range(n)}
    anti = {matrix[i][n - 1 - i] for i in range(n)}
    return check_valid_zip(matrix) and main == target and anti == target

print(check_valid_with_diagonals([[1, 2, 3], [3, 1, 2], [2, 3, 1]]))   # chéo chính toàn 1 -> False
print(check_valid_with_diagonals([[1, 2, 3, 4], [3, 4, 1, 2], [4, 3, 2, 1], [2, 1, 4, 3]]))

False
True


### 15.2. Trả về **hàng/cột đầu tiên** bị lỗi (để báo cho người dùng)

In [30]:
def first_invalid_line(matrix):
    n = len(matrix)
    target = set(range(1, n + 1))
    for i, row in enumerate(matrix):
        if set(row) != target:
            return ("row", i, sorted(target - set(row)))
    for j, col in enumerate(zip(*matrix)):
        if set(col) != target:
            return ("col", j, sorted(target - set(col)))
    return None

print(first_invalid_line([[1, 2, 3], [1, 2, 3], [3, 1, 2]]))
print(first_invalid_line([[1, 2, 3], [3, 1, 2], [2, 3, 1]]))

('col', 0, [2])
None


Ở đây `target - set(col)` là **hiệu hai set**: các số có trong target nhưng thiếu trong cột.

### 15.3. Giá trị không giới hạn — dùng `Counter`

Nếu đề nói "mỗi hàng chứa mỗi số trong `1..n` **đúng một lần**" nhưng ô có thể ngoài khoảng, `Counter(row) == Counter(range(1, n+1))` là cách so sánh "đa tập" chắc chắn.

In [31]:
from collections import Counter
print(Counter([3, 1, 2]) == Counter(range(1, 4)))
print(Counter([1, 2, 5]) == Counter(range(1, 4)))

True
False


## 16. Bài tập (TODO — tự làm, đáp án ở cuối)

### Bài 1 — Đoán output (không chạy)
Cho `m = [[2, 1], [2, 1]]`. Hỏi:
- `len(set(m[0]))` = ?
- `list(zip(*m))` = ?
- `checkValid(m)` = ?

### Bài 2 — Trace bằng tay
Lập bảng trace (như mục TRACE) cho `[[1,2,3],[2,3,1],[3,1,1]]`. Hàm dừng ở bước mấy, ở hàng hay cột?

### Bài 3 — Điền chỗ trống: kiểm tra một dãy bằng mảng đánh dấu (không dùng set)

In [32]:
def exercise_is_permutation(seq: list[int], n: int) -> bool:
    # TODO 1: tạo seen = [False] * (n + 1)
    # TODO 2: với mỗi v trong seq: nếu v ngoài 1..n hoặc seen[v] đã True -> return False
    # TODO 3: đánh dấu seen[v] = True
    # TODO 4: return True
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_is_permutation([3, 1, 2], 3) == True
# assert exercise_is_permutation([1, 1, 3], 3) == False
# assert exercise_is_permutation([1, 2, 5], 3) == False

### Bài 4 — Tự viết `checkValid` chỉ dùng **một** vòng lặp đôi `for i ... for j ...` với set
Gợi ý cấu trúc: `rows = [set() for _ in range(n)]`, `cols = [set() for _ in range(n)]`.

In [33]:
def exercise_check_valid_one_pass(matrix: list[list[int]]) -> bool:
    # TODO: dùng rows[i], cols[j] là các set; phát hiện trùng khi thêm
    pass

# assert exercise_check_valid_one_pass([[1, 2, 3], [3, 1, 2], [2, 3, 1]]) == True
# assert exercise_check_valid_one_pass([[1, 2, 3], [1, 2, 3], [3, 1, 2]]) == False

### Bài 5 — Biến thể: đếm số hàng hợp lệ và số cột hợp lệ
Trả về tuple `(so_hang_hop_le, so_cot_hop_le)`.

In [34]:
def exercise_count_valid_lines(matrix: list[list[int]]) -> tuple[int, int]:
    # TODO
    pass

# assert exercise_count_valid_lines([[1, 2, 3], [1, 2, 3], [3, 1, 2]]) == (3, 0)

### Bài 6 — Mini challenge: Sudoku 4x4
Ma trận `4x4`, giá trị `1..4`. Hợp lệ khi mọi hàng, cột và **4 ô vuông 2x2** (góc trên-trái, trên-phải, dưới-trái, dưới-phải) đều đủ `1..4`.

In [35]:
def exercise_valid_sudoku_4x4(grid: list[list[int]]) -> bool:
    # TODO: tái sử dụng ý tưởng set; ô vuông có góc trên-trái (r0, c0) với r0, c0 thuộc {0, 2}
    pass

# assert exercise_valid_sudoku_4x4([[1,2,3,4],[3,4,1,2],[2,1,4,3],[4,3,2,1]]) == True
# assert exercise_valid_sudoku_4x4([[1,2,3,4],[2,3,4,1],[3,4,1,2],[4,1,2,3]]) == False

### Gợi ý 3 mức (cho mục bài tập)

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | Set bỏ trùng; zip lật hàng thành cột | `set([2,1])`, `zip([2,1],[2,1])` | cột 0 là `(2,2)` |
| 2 | Hàng trước, cột sau | Hàng 2 là `[3,1,1]` | dừng ở hàng 2, set `{1,3}` |
| 3 | Chỉ số = giá trị | `if v < 1 or v > n or seen[v]` | `seen[v] = True` sau khi kiểm |
| 4 | Mỗi hàng, mỗi cột một set | `v in rows[i] or v in cols[j]` | `rows[i].add(v); cols[j].add(v)` |
| 5 | Đếm thay vì return sớm | `sum(1 for row in ... if ...)` | dùng `zip(*matrix)` cho cột |
| 6 | Thêm loại "dãy" thứ ba: khối 2x2 | `for r0 in (0,2): for c0 in (0,2)` | khối = `{grid[r0+a][c0+b] for a in (0,1) for b in (0,1)}` |

## 17. Đáp án tham khảo (tự làm trước rồi hãy xem)

**Bài 1:** `len(set([2,1])) = 2`; `list(zip(*m)) = [(2, 2), (1, 1)]`; `checkValid(m) = False` (cột 0 toàn số 2).

**Bài 2:** Hàng 0 `{1,2,3}` ok → hàng 1 `{1,2,3}` ok → hàng 2 `[3,1,1]` set `{1,3}` len 2 → **dừng ở bước 3, tại hàng 2**, trả `False`.

In [36]:
m = [[2, 1], [2, 1]]
print(len(set(m[0])), list(zip(*m)), Solution().checkValid(m))
trace_check_valid([[1, 2, 3], [2, 3, 1], [3, 1, 1]])

2 [(2, 2), (1, 1)] False
Bước 1: hàng 0 = [1, 2, 3] -> set = [1, 2, 3], len = 3
Bước 2: hàng 1 = [2, 3, 1] -> set = [1, 2, 3], len = 3
Bước 3: hàng 2 = [3, 1, 1] -> set = [1, 3], len = 2
  -> thiếu số, return False


False

In [37]:
def answer_is_permutation(seq, n):
    seen = [False] * (n + 1)
    for v in seq:
        if v < 1 or v > n or seen[v]:
            return False
        seen[v] = True
    return True


def answer_check_valid_one_pass(matrix):
    n = len(matrix)
    rows = [set() for _ in range(n)]
    cols = [set() for _ in range(n)]
    for i in range(n):
        for j in range(n):
            v = matrix[i][j]
            if v in rows[i] or v in cols[j]:
                return False
            rows[i].add(v)
            cols[j].add(v)
    return True


def answer_count_valid_lines(matrix):
    n = len(matrix)
    target = set(range(1, n + 1))
    good_rows = sum(1 for row in matrix if set(row) == target)
    good_cols = sum(1 for col in zip(*matrix) if set(col) == target)
    return (good_rows, good_cols)


def answer_valid_sudoku_4x4(grid):
    target = {1, 2, 3, 4}
    lines = list(grid) + list(zip(*grid))
    for r0 in (0, 2):
        for c0 in (0, 2):
            lines.append([grid[r0 + a][c0 + b] for a in (0, 1) for b in (0, 1)])
    return all(set(line) == target for line in lines)


assert answer_is_permutation([3, 1, 2], 3) == True
assert answer_is_permutation([1, 1, 3], 3) == False
assert answer_is_permutation([1, 2, 5], 3) == False
assert answer_check_valid_one_pass([[1, 2, 3], [3, 1, 2], [2, 3, 1]]) == True
assert answer_check_valid_one_pass([[1, 2, 3], [1, 2, 3], [3, 1, 2]]) == False
assert answer_count_valid_lines([[1, 2, 3], [1, 2, 3], [3, 1, 2]]) == (3, 0)
assert answer_valid_sudoku_4x4([[1,2,3,4],[3,4,1,2],[2,1,4,3],[4,3,2,1]]) == True
assert answer_valid_sudoku_4x4([[1,2,3,4],[2,3,4,1],[3,4,1,2],[4,1,2,3]]) == False
for name, m, exp in cases:
    assert answer_check_valid_one_pass(m) == exp, name
print("Đáp án tham khảo: PASS toàn bộ test!")

Đáp án tham khảo: PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def checkValid(self, matrix: list[list[int]]) -> bool:
        target = set(range(1, len(matrix) + 1))
        return all(set(r) == target for r in matrix) and \
               all(set(c) == target for c in zip(*matrix))
```

**Khung tư duy:** "đủ 1..n / không trùng" → set (hoặc mảng đánh dấu) → kiểm từng *dãy* (hàng, cột, khối…) → dừng sớm khi hỏng.

**Checklist trước khi nộp:**
- [ ] Đã kiểm tra **cả hàng và cột** chưa?
- [ ] Lấy cột đúng `matrix[r][j]` (hoặc `zip(*matrix)`)?
- [ ] Không dùng tổng để thay cho kiểm tra tập hợp?
- [ ] Nếu dùng `len(set) == n`, có dựa vào constraint `1 <= x <= n` không? (Có — bài này OK.)
- [ ] `zip(*matrix)` có bị dùng lại hai lần khi là iterator không?
- [ ] Test `n = 1`, hàng đúng-cột sai, ô sai ở cuối.